In [ ]:
# !pip install nltk scikit-learn textblob vaderSentiment wordcloud matplotlib tensorflow
import re, string
import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, quiet=True)

In [ ]:
paragraph = ("Football is more than just a game for millions of people around the world. "
             "Every four years, the World Cup brings together nations, cultures, and rivalries on one stage. "
             "Players like Messi and Ronaldo have inspired young children to practice on dusty streets every evening. "
             "Modern football also relies heavily on data analytics, tactics, and fitness science to gain an edge. "
             "Fans don't just watch the matches; they live every goal, every save, and every heartbreak. "
             "That passion is exactly why football is called the beautiful game!")
print(paragraph)

In [ ]:
clean = re.sub(r'[^\w\s]', '', paragraph.lower())
print(clean)

In [ ]:
Tokenize
from nltk.tokenize import word_tokenize, sent_tokenize
sentences = sent_tokenize(paragraph)
words = word_tokenize(clean)
print("Sentences:", len(sentences))
for s in sentences: print(" -", s)
print("\nWords:", words)

In [ ]:
# split() vs word_tokenize()
split_tokens = paragraph.split()
nltk_tokens = word_tokenize(paragraph)
print("split()        :", len(split_tokens), split_tokens[:15])
print("word_tokenize():", len(nltk_tokens), nltk_tokens[:15])
print("\nOnly in split():", [t for t in split_tokens if t not in nltk_tokens][:10])
print("Only in NLTK   :", [t for t in nltk_tokens if t not in split_tokens][:10])

In [ ]:
#Remove stopwords
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
filtered = [w for w in words if w not in stop_words]
print(filtered)

In [ ]:
# Word frequency distribution
from nltk.probability import FreqDist
import matplotlib.pyplot as plt
fdist = FreqDist(filtered)
print(fdist.most_common(10))
plt.figure(figsize=(10, 4))
fdist.plot(15, title="Top 15 words (stopwords removed)")
plt.show()

In [ ]:
# Alphabet-only words re.findall
alpha_words = re.findall(r'\b[a-zA-Z]+\b', paragraph.lower())
print(alpha_words)

In [ ]:
# Remove stopwords
alpha_nostop = [w for w in alpha_words if w not in stop_words]
print(alpha_nostop)

In [ ]:
# Porter stemming
from nltk.stem import PorterStemmer, WordNetLemmatizer
ps = PorterStemmer()
stemmed = [ps.stem(w) for w in alpha_nostop]
print(stemmed)

In [ ]:
# WordNet lemmatization
lem = WordNetLemmatizer()
lemmatized = [lem.lemmatize(w) for w in alpha_nostop]
lemmatized_v = [lem.lemmatize(w, pos='v') for w in alpha_nostop]
print(lemmatized)

In [ ]:
# Compare
import pandas as pd
df = pd.DataFrame({'original': alpha_nostop, 'stemmed': stemmed,
                   'lemma (noun)': lemmatized, 'lemma (verb)': lemmatized_v})
print(df[(df.original != df.stemmed) | (df.original != df['lemma (verb)'])].to_string(index=False))

## Bag of Words and TF-IDF

In [ ]:
texts = [
    "The new smartphone has an amazing camera and a long battery life.",
    "The government announced a new policy to improve public transport and reduce traffic.",
    "The team won the championship after a thrilling final match and a brilliant goal."
]

In [ ]:
# Bag of Words
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
cv = CountVectorizer(stop_words='english')
bow = cv.fit_transform(texts)
bow_df = pd.DataFrame(bow.toarray(), columns=cv.get_feature_names_out(), index=['Text1', 'Text2', 'Text3'])
print(bow_df.T.to_string())

In [ ]:
# TF-IDF + top 3 keywords
tfidf = TfidfVectorizer(stop_words='english')
X = tfidf.fit_transform(texts)
features = tfidf.get_feature_names_out()
tfidf_df = pd.DataFrame(X.toarray(), columns=features, index=['Text1', 'Text2', 'Text3'])
print(tfidf_df.round(3).T.to_string())

import numpy as np
for i in range(len(texts)):
    row = X[i].toarray().ravel()
    top = np.argsort(row)[::-1][:3]
    print(f"\nText {i+1} top 3 keywords:", [(features[j], round(float(row[j]), 3)) for j in top])

In [ ]:
tech1 = ("Artificial intelligence is the simulation of human intelligence by machines. "
         "It uses data and algorithms to learn patterns and make decisions. "
         "Machine learning and neural networks power applications like speech recognition and image analysis. "
         "AI is transforming healthcare, finance, and education.")
tech2 = ("Blockchain is a distributed ledger that records transactions across many computers. "
         "It uses cryptography and consensus to keep data secure and tamper proof. "
         "Cryptocurrencies like Bitcoin and smart contracts run on blockchain networks. "
         "Blockchain is transforming finance, supply chains, and digital identity.")

In [ ]:
# Preprocess + tokenize
def preprocess(t):
    t = re.sub(r'[^a-z\s]', '', t.lower())
    return [w for w in word_tokenize(t) if w not in stop_words]
t1, t2 = preprocess(tech1), preprocess(tech2)
print(t1); print(t2)

In [ ]:
# Jaccard
s1, s2 = set(t1), set(t2)
jaccard = len(s1 & s2) / len(s1 | s2)
print("Common words:", s1 & s2)
print("Jaccard similarity:", round(jaccard, 4))

In [ ]:
# Cosine with TF-IDF
from sklearn.metrics.pairwise import cosine_similarity
vec = TfidfVectorizer(stop_words='english')
M = vec.fit_transform([tech1, tech2])
cos = cosine_similarity(M[0], M[1])[0][0]
print("Cosine similarity:", round(cos, 4))

In [ ]:
reviews = [
    "Absolutely love this phone! The camera is fantastic and the battery lasts all day.",
    "Terrible service. The delivery was late and the support staff was rude.",
    "The package arrived on Tuesday. The product has a black finish and comes with a charger.",
    "Great value for money, excellent build quality and super fast performance.",
    "Worst purchase ever. It stopped working after two days and I am very disappointed.",
    "Decent laptop, average display but the keyboard is comfortable.",
    "Amazing experience! Friendly staff, quick delivery, and a beautiful packaging.",
    "The headphones are awful, the sound is poor and they feel cheap."
]

In [ ]:
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()

rows = []
for r in reviews:
    tb = TextBlob(r)
    comp = vader.polarity_scores(r)['compound']
    label = 'Positive' if comp >= 0.05 else 'Negative' if comp <= -0.05 else 'Neutral'
    rows.append([r[:50] + '...', round(tb.sentiment.polarity, 3), round(tb.sentiment.subjectivity, 3), comp, label])
res = pd.DataFrame(rows, columns=['Review', 'TB polarity', 'TB subjectivity', 'VADER compound', 'Class'])
print(res.to_string(index=False))

In [ ]:
# Word cloud of positive reviews
from wordcloud import WordCloud
pos_text = " ".join(r for r, l in zip(reviews, res['Class']) if l == 'Positive')
wc = WordCloud(width=800, height=400, background_color='white', stopwords=stop_words).generate(pos_text)
plt.figure(figsize=(10, 5))
plt.imshow(wc, interpolation='bilinear'); plt.axis('off')
plt.title("Word Cloud - Positive Reviews"); plt.show()

In [ ]:
import numpy as np
try:
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
except ImportError:
    from keras.legacy.preprocessing.text import Tokenizer
    from keras.utils import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

data = ("Football is a game of passion and skill. Every player trains hard to become a champion. "
        "The fans fill the stadium and cheer for their team. A single goal can change the whole match. "
        "Teamwork and discipline help a team win the trophy. The coach plans the tactics and the players "
        "follow the plan. Football brings people together and gives them joy. The beautiful game never "
        "fails to inspire young players around the world.")

In [ ]:
# Tokenize
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])
total_words = len(tokenizer.word_index) + 1
print("Vocabulary size:", total_words)

# Create n-gram input sequences (per sentence)
sequences = []
for line in data.split('.'):
    line = line.strip()
    if not line: continue
    tokens = tokenizer.texts_to_sequences([line])[0]
    for i in range(2, len(tokens) + 1):
        sequences.append(tokens[:i])
max_len = max(len(s) for s in sequences)
sequences = pad_sequences(sequences, maxlen=max_len, padding='pre')
X, y = sequences[:, :-1], sequences[:, -1]
y = to_categorical(y, num_classes=total_words)
print("X:", X.shape, "y:", y.shape)

In [ ]:
# Build model
model = Sequential([
    Embedding(total_words, 50),
    LSTM(100),
    Dense(total_words, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.build(input_shape=(None, max_len - 1))
model.summary()

In [ ]:
# Train
history = model.fit(X, y, epochs=200, verbose=0)
print("Final accuracy:", round(history.history['accuracy'][-1], 3))

In [ ]:
# Generate text from a seed word
def generate(seed, n_words=8):
    text = seed
    for _ in range(n_words):
        seq = tokenizer.texts_to_sequences([text])[0]
        seq = pad_sequences([seq], maxlen=max_len - 1, padding='pre')
        pred = np.argmax(model.predict(seq, verbose=0), axis=-1)[0]
        word = tokenizer.index_word.get(pred, '')
        if not word: break
        text += ' ' + word
    return text

for seed in ["football", "the fans", "teamwork"]:
    print(generate(seed))